# 02_Silver_Cleaning

Run on Databricks serverless with the uploaded Olist files. Original team work: review, execute, validate and document results.

In [ ]:
CATALOG = "workspace"
SCHEMA = "ecommerce_bigdata"
VOLUME = "raw_data"
RAW_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print('Raw data path:', RAW_PATH)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
prefix=f"{CATALOG}.{SCHEMA}"
def bronze(name): return spark.table(f"{prefix}.bronze_{name}")
def save(df,name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{prefix}.silver_{name}")
    print('Saved', name, df.count())
orders=(bronze('orders').dropDuplicates(['order_id'])
    .withColumn('purchase_at',F.to_timestamp('order_purchase_timestamp'))
    .withColumn('delivered_at',F.to_timestamp('order_delivered_customer_date'))
    .withColumn('estimated_at',F.to_timestamp('order_estimated_delivery_date'))
    .filter(F.col('order_id').isNotNull() & F.col('customer_id').isNotNull()))
save(orders,'orders')
customers=bronze('customers').dropDuplicates(['customer_id']).filter('customer_id IS NOT NULL')
save(customers,'customers')
items=(bronze('order_items').dropDuplicates(['order_id','order_item_id'])
    .withColumn('price',F.col('price').cast('double'))
    .withColumn('freight_value',F.col('freight_value').cast('double'))
    .filter('order_id IS NOT NULL AND product_id IS NOT NULL AND price >= 0 AND freight_value >= 0'))
save(items,'order_items')
payments=(bronze('order_payments').dropDuplicates(['order_id','payment_sequential'])
    .withColumn('payment_value',F.col('payment_value').cast('double'))
    .withColumn('payment_installments',F.col('payment_installments').cast('int'))
    .filter('order_id IS NOT NULL AND payment_value >= 0'))
save(payments,'order_payments')
products=bronze('products').dropDuplicates(['product_id'])
translation=bronze('category_translation').dropDuplicates(['product_category_name'])
products=products.join(F.broadcast(translation.select('product_category_name','product_category_name_english')),'product_category_name','left')
save(products,'products')
reviews=(bronze('order_reviews').withColumn('review_score',F.col('review_score').cast('int'))
    .filter('order_id IS NOT NULL AND review_score BETWEEN 1 AND 5')
    .groupBy('order_id').agg(F.avg('review_score').alias('review_score')))
save(reviews,'order_reviews')
save(bronze('sellers').dropDuplicates(['seller_id']),'sellers')